# SegFormer fine-tune on Colab — 차선 세그 (CVAT zip 업로드)

Phase 1: 차선 세그용 SegFormer fine-tune.
**3 클래스 + 배경** → id2label = {0: background, 1: left-solid, 2: right-solid, 3: center-dashed}

이 id2label 순서는 `data_pipeline/extract_labels.py`의 `SegFormerLaneSeg`가
기대하는 것과 정확히 일치해야 한다 (배경=0, 의미 클래스 1..3 → 출력 채널 0..2).

**라벨링: CVAT**. CVAT에서 polygon 세그 라벨링 후 **Segmentation mask 1.1**로
export → zip을 이 노트북에 업로드한다 (Drive 불필요).

사용 순서:
1. **런타임 → GPU (T4/L4)**
2. CVAT에서 차선 polygon 라벨링(좌실선/우실선/중앙점선) →
   **Export → `Segmentation mask 1.1`** → zip 다운로드
3. 아래 **업로드 셀**에서 zip 업로드 (`SegmentationClass/` PNG + `labelmap.txt` 구조)
4. `CONFIG`의 `CVAT_LABEL2ID`를 labelmap의 클래스명과 맞추고 실행
5. 끝나면 체크포인트 폴더(`segformer_lane/`)를 zip으로 받아 압축해제 후
   `extract_labels.py --segformer_ckpt segformer_lane` 로 사용

> CVAT `Segmentation mask 1.1` export 구조:
> - `ImageSets/`, `JPEGImages/`(원본), `SegmentationClass/`(색 마스크 PNG),
>   `labelmap.txt`(`name:R,G,B:...` 클래스→색 매핑).
> 아래 셀이 labelmap의 색을 우리 클래스 id(0..3)로 변환한다.

## 1. 설치

In [ ]:
!pip install -q -U transformers datasets evaluate
import torch, transformers
print('transformers', transformers.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 2. CONFIG

`ID2LABEL` 순서 고정 — extract_labels.py contract (배경 0, 좌실선 1, 우실선 2, 중앙점선 3).
`CVAT_LABEL2ID`는 CVAT labelmap.txt의 **클래스 이름**을 이 id로 매핑한다 (이름이 다르면
여기만 수정).

In [ ]:
BASE_MODEL = 'nvidia/segformer-b0-finetuned-ade-512-512'  # b0 = Jetson 가벼움
IMG_SIZE   = 512        # SegFormer 학습 입력 (추론은 224 lane에 upsample되므로 무관)
EPOCHS     = 80
BATCH      = 8
LR         = 6e-5
OUT_DIR    = '/content/segformer_lane'
DATA_ROOT  = '/content/cvat_seg'   # 업로드한 zip을 풀 위치
VAL_FRAC   = 0.2                   # train/val split (CVAT는 split 없이 한 덩어리)

# extract_labels.SegFormerLaneSeg와 반드시 일치
ID2LABEL = {0: 'background', 1: 'left-solid', 2: 'right-solid', 3: 'center-dashed'}
LABEL2ID = {v: k for k, v in ID2LABEL.items()}
NUM_LABELS = len(ID2LABEL)

# CVAT labelmap.txt의 클래스명 → 우리 id. CVAT에서 라벨 이름을 아래와 다르게 지었으면
# 여기만 맞추면 된다 (background는 CVAT가 자동 0으로 둠).
CVAT_LABEL2ID = {
    'left-solid':    1,
    'right-solid':   2,
    'center-dashed': 3,
}
print('id2label:', ID2LABEL)

## 3. CVAT zip 업로드 + 압축해제

In [ ]:
import os, glob, zipfile, shutil
from google.colab import files

shutil.rmtree(DATA_ROOT, ignore_errors=True)
os.makedirs(DATA_ROOT, exist_ok=True)
up = files.upload()                      # CVAT 'Segmentation mask 1.1' zip 선택
zip_name = next(iter(up))
with zipfile.ZipFile(zip_name) as z:
    z.extractall(DATA_ROOT)

# CVAT export는 하위 폴더로 한 단계 더 들어갈 수 있어 실제 루트를 찾는다
def find_dir(root, name):
    hits = glob.glob(os.path.join(root, '**', name), recursive=True)
    return os.path.dirname(hits[0]) if hits else None

SEG_ROOT = find_dir(DATA_ROOT, 'labelmap.txt')
assert SEG_ROOT, 'labelmap.txt 없음 — CVAT Segmentation mask 1.1 export인지 확인'
IMG_DIR  = os.path.join(SEG_ROOT, 'JPEGImages')
MASK_DIR = os.path.join(SEG_ROOT, 'SegmentationClass')
print('SEG_ROOT :', SEG_ROOT)
print('images   :', len(glob.glob(os.path.join(IMG_DIR, '*'))))
print('masks    :', len(glob.glob(os.path.join(MASK_DIR, '*'))))

## 4. labelmap 파싱 + 색 마스크 → 클래스 인덱스 변환

CVAT `SegmentationClass/`는 **색(RGB) 마스크**다. `labelmap.txt`(`name:R,G,B::`)를
읽어 각 클래스 색을 우리 id(0..3)로 바꾸는 룩업을 만든다.

In [ ]:
import numpy as np

# labelmap.txt 형식:  name:R,G,B:: (background 포함, 줄마다 한 클래스)
color2id = {}
with open(os.path.join(SEG_ROOT, 'labelmap.txt')) as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith('#'):
            continue
        parts = line.split(':')
        name = parts[0].strip()
        rgb  = tuple(int(x) for x in parts[1].split(','))
        if name in ('background', '_background_'):
            color2id[rgb] = 0
        elif name in CVAT_LABEL2ID:
            color2id[rgb] = CVAT_LABEL2ID[name]
        # labelmap에만 있고 우리가 안 쓰는 클래스는 배경(0)으로 떨어뜨림
print('color2id:', color2id)
assert any(v != 0 for v in color2id.values()), \
    'CVAT 클래스명과 CVAT_LABEL2ID가 안 맞음 — labelmap.txt의 이름 확인'

def rgb_mask_to_index(mask_rgb: np.ndarray) -> np.ndarray:
    """(H,W,3) 색 마스크 → (H,W) 클래스 인덱스맵 (0..3). 매핑 없는 색은 0."""
    idx = np.zeros(mask_rgb.shape[:2], dtype=np.uint8)
    for color, cid in color2id.items():
        if cid == 0:
            continue
        hit = np.all(mask_rgb == np.array(color, dtype=mask_rgb.dtype), axis=-1)
        idx[hit] = cid
    return idx

In [ ]:
import glob, random
from PIL import Image
import torch
from torch.utils.data import Dataset
from transformers import SegformerImageProcessor

def list_pairs():
    """(image_path, mask_path) 쌍. CVAT: JPEGImages/<stem>.jpg ↔ SegmentationClass/<stem>.png"""
    pairs = []
    for img in sorted(glob.glob(os.path.join(IMG_DIR, '*'))):
        stem = os.path.splitext(os.path.basename(img))[0]
        mask = os.path.join(MASK_DIR, stem + '.png')
        if os.path.exists(mask):
            pairs.append((img, mask))
    return pairs

all_pairs = list_pairs()
assert all_pairs, 'no (image, mask) pairs — JPEGImages/SegmentationClass 파일명 확인'
random.Random(0).shuffle(all_pairs)
n_val = max(1, int(len(all_pairs) * VAL_FRAC))
val_pairs, train_pairs = all_pairs[:n_val], all_pairs[n_val:]
print('train pairs:', len(train_pairs), '| val pairs:', len(val_pairs))

processor = SegformerImageProcessor.from_pretrained(BASE_MODEL, do_reduce_labels=False)
processor.size = {'height': IMG_SIZE, 'width': IMG_SIZE}

class LaneSegDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, i):
        img_path, mask_path = self.pairs[i]
        image = Image.open(img_path).convert('RGB')
        mask_rgb = np.array(Image.open(mask_path).convert('RGB'))
        mask_idx = rgb_mask_to_index(mask_rgb)            # (H,W) 0..3
        enc = processor(image, Image.fromarray(mask_idx), return_tensors='pt')
        return {k: v.squeeze(0) for k, v in enc.items()}

train_ds = LaneSegDataset(train_pairs)
val_ds   = LaneSegDataset(val_pairs)

## 5. 모델 + Trainer

In [ ]:
from transformers import SegformerForSemanticSegmentation, TrainingArguments, Trainer

model = SegformerForSemanticSegmentation.from_pretrained(
    BASE_MODEL,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,   # ADE20K head → 4클래스 head 교체
)

args = TrainingArguments(
    output_dir='/content/seg_ckpts',
    learning_rate=LR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=1,
    load_best_model_at_end=True,
    logging_steps=10,
    remove_unused_columns=False,
    fp16=torch.cuda.is_available(),
    report_to='none',
)

In [ ]:
import evaluate
metric = evaluate.load('mean_iou')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    logits = torch.tensor(logits)
    # logits: (B, C, h, w) → 라벨 해상도로 upsample 후 argmax
    upsampled = torch.nn.functional.interpolate(
        logits, size=labels.shape[-2:], mode='bilinear', align_corners=False)
    preds = upsampled.argmax(dim=1).numpy()
    res = metric.compute(predictions=preds, references=labels,
                         num_labels=NUM_LABELS, ignore_index=255,
                         reduce_labels=False)
    return {'mean_iou': res['mean_iou'], 'mean_acc': res['mean_accuracy']}

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)
trainer.train()

## 6. 체크포인트 저장 (extract_labels.py가 from_pretrained로 로드)

`model.save_pretrained` + `processor.save_pretrained`를 같은 폴더에 저장해야
`SegFormerLaneSeg(checkpoint_path)`가 그대로 로드한다.

In [ ]:
import shutil
model.save_pretrained(OUT_DIR)
processor.save_pretrained(OUT_DIR)
print('saved checkpoint ->', OUT_DIR)
print(os.listdir(OUT_DIR))   # config.json, model.safetensors, preprocessor_config.json 등

# zip으로 묶어 다운로드 (Drive 안 씀)
shutil.make_archive('/content/segformer_lane', 'zip', OUT_DIR)
from google.colab import files
files.download('/content/segformer_lane.zip')

## 7. (선택) sanity check — 한 장 추론 시각화

In [ ]:
import matplotlib.pyplot as plt
model.eval()
img_path, _ = val_pairs[0]
image = Image.open(img_path).convert('RGB')
inp = processor(image, return_tensors='pt').to(model.device)
with torch.no_grad():
    logits = model(**inp).logits
up = torch.nn.functional.interpolate(logits, size=image.size[::-1],
                                     mode='bilinear', align_corners=False)
pred = up.argmax(1)[0].cpu().numpy()
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(image); ax[0].set_title('lane'); ax[0].axis('off')
ax[1].imshow(pred, vmin=0, vmax=NUM_LABELS-1, cmap='tab10')
ax[1].set_title('pred (0 bg / 1 L / 2 R / 3 dash)'); ax[1].axis('off')
plt.show()